# IDR Plans Wealth Simulation
## Income-Driven Repayment Analysis — With MOE/SE & Sensitivity Analysis

**Author:** Oscar Mayorga | Equity Research Cooperative (EqRC)  
**Updated:** April 2026  

### Overview
This notebook simulates net worth accumulation from college graduation (age 22) to retirement (age 62) across six federal Income-Driven Repayment (IDR) plans. The simulation incorporates:

- **Updated 2025–2026 data** (BLS Q4 2024, HHS 2026 FPL, Freddie Mac PMMS April 2026)
- **Margin of Error / Standard Error** integration — all stochastic parameters are drawn from normal distributions parameterized by their reported point estimates and SE/MOE values
- **Sensitivity Analysis** — tornado chart (one-at-a-time parameter perturbation) and economic scenario matrix (pessimistic / baseline / optimistic)

### Data Sources
| Parameter | Source | Link |
|---|---|---|
| Income by race/gender | BLS Usual Weekly Earnings Q4 2024 | [BLS Q4 2024](https://www.bls.gov/news.release/archives/wkyeng_02212025.pdf) |
| FPL thresholds | HHS 2026 Poverty Guidelines | [HHS 2026](https://aspe.hhs.gov/topics/poverty-economic-mobility/poverty-guidelines) |
| Homeownership rates | Census CPS/HVS 2025 via FRED | [FRED](https://fred.stlouisfed.org/series/BOAAAHORUSQ156N) |
| Student loan debt | Education Data Initiative 2025 | [EDI 2025](https://educationdata.org/average-debt-for-a-bachelors-degree) |
| Mortgage rate | Freddie Mac PMMS April 2, 2026 | [Freddie Mac](https://www.freddiemac.com/pmms) |
| Family income | Census CPS/ASEC 2024 | [Census](https://www.census.gov/topics/income-poverty/income.html) |
| Employment rates | BLS CPS Tables 3/4, 2025 Annual | [BLS CPS](https://www.bls.gov/cps/tables.htm) |

> **Note:** Q4 2025 BLS earnings were not produced due to the October 2025 federal government shutdown. Q4 2024 is the most recent complete quarter available.


## Setup — Imports & Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# Display charts inline
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

# ── Output directory ──
# Change this to wherever you want to save the PNG files.
# Set to None to skip saving (charts will still display inline).
output_dir = r"C:\Users\Oscar\Dropbox\Downloads"

if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

print(f"Charts will be saved to: {output_dir}")
print(f"Charts will also display inline in this notebook.")


## Parameters — All Variables with MOE/SE

In [ ]:
# ── RANDOM SEED for reproducibility ──
np.random.seed(42)

# ── 2026 HHS Federal Poverty Guidelines ──
# Source: https://www.medicaidplanningassistance.org/federal-poverty-guidelines/
fpl_single      = 15_960.0   # 2026 FPL, 1-person household (was $15,650 in 2025)
fpl_family_of_4 = 33_000.0   # 2026 FPL, 4-person household (was $32,150 in 2025)

# ── Inflation ──
inflation_rate  = 0.025      # Federal Reserve long-run target
stage_durations = [8, 10, 10, 12]  # years per career stage

# ── Income by race/gender — BLS Q4 2024 Annualized (weekly × 52) ──
# SE values from BLS/Census Variable Documentation Table
# Note: Q4 2025 data unavailable due to Oct 2025 federal gov shutdown
data = {
    'Black Men':    {'avg_income': 58_136.0, 'income_se': 520.0},
    'Black Women':  {'avg_income': 50_856.0, 'income_se': 312.0},
    'White Men':    {'avg_income': 68_692.0, 'income_se': 312.0},
    'White Women':  {'avg_income': 56_888.0, 'income_se': 260.0},
    'Latinx Men':   {'avg_income': 52_052.0, 'income_se': 364.0},
    'Latinx Women': {'avg_income': 43_888.0, 'income_se': 312.0},
}

# ── Homeownership Rates — Census CPS/HVS 2025 (FRED) ──
# MOE: ±1.3 pp Black & Hispanic, ±0.5 pp White (Census HVS documentation)
home_purchase_rates = {
    'Black Men':    0.439, 'Black Women':  0.439,
    'White Men':    0.740, 'White Women':  0.740,
    'Latinx Men':   0.487, 'Latinx Women': 0.487,
}
home_purchase_rates_moe = {
    'Black Men':    0.013, 'Black Women':  0.013,
    'White Men':    0.005, 'White Women':  0.005,
    'Latinx Men':   0.013, 'Latinx Women': 0.013,
}

home_purchase_rates_by_race     = {'Black': 0.439, 'White': 0.740, 'Hispanic': 0.487}
home_purchase_rates_moe_by_race = {'Black': 0.013, 'White': 0.005, 'Hispanic': 0.013}

# ── Employment Rates — BLS CPS Table 3/4, 2025 Annual Averages ──
# By career stage [22-30, 30-40, 40-50, 50-62]; MOE ±1–2 pp
employment_rates = {
    'Black Men':    [0.78, 0.82, 0.77, 0.62],
    'Black Women':  [0.71, 0.76, 0.75, 0.58],
    'White Men':    [0.87, 0.89, 0.86, 0.71],
    'White Women':  [0.76, 0.76, 0.75, 0.60],
    'Latinx Men':   [0.86, 0.89, 0.86, 0.73],
    'Latinx Women': [0.71, 0.70, 0.69, 0.58],
}
employment_rates_se = 0.015   # ±1.5 pp (midpoint of ±1–2 pp range)

employment_rates_by_race = {
    'Black':    [0.745, 0.790, 0.760, 0.600],
    'White':    [0.815, 0.825, 0.805, 0.655],
    'Hispanic': [0.785, 0.795, 0.775, 0.655],
}

# ── Family Income — Census CPS/ASEC 2023-2024 ──
family_income_by_race = {'Black': 56_020.0, 'White': 92_530.0, 'Hispanic': 70_950.0}
family_income_moe     = {'Black': 323.0,    'White': 224.0,    'Hispanic': 451.0}

family_income_tiers = {
    'Median':           1.00,
    '25% Above Median': 1.25,
    '50% Above Median': 1.50,
}

income_brackets = ['Lower 25%', 'Median 50%', 'Upper 25%']
income_factors  = [0.75, 1.0, 1.25]

num_individuals = 1_000_000

# ── IDR Plans ──
# NOTE: SAVE is court-blocked; ICR/PAYE phasing out (new enrollment closes July 2028)
idr_plans = {
    'IBR_2014':       {'repayment_rate': 0.10, 'years': 20, 'fpl_multiplier': 1.50},
    'IBR_pre2014':    {'repayment_rate': 0.15, 'years': 25, 'fpl_multiplier': 1.50},
    'ICR':            {'repayment_rate': 0.20, 'years': 25, 'fpl_multiplier': 1.00},
    'PAYE':           {'repayment_rate': 0.10, 'years': 20, 'fpl_multiplier': 1.50},
    'SAVE_undergrad': {'repayment_rate': 0.05, 'years': 20, 'fpl_multiplier': 2.25},
    'SAVE_grad':      {'repayment_rate': 0.10, 'years': 25, 'fpl_multiplier': 2.25},
}

# ── Financial Parameters ──
initial_student_loan_debt    = 37_500.0   # Education Data Initiative 2025
initial_student_loan_debt_se = 2_000.0
student_loan_interest_rate   = 0.0653     # Federal rate AY 2024-25

home_appreciation_rate_nominal   = 0.030  # FHFA HPI long-run average
personal_asset_growth_rate_nominal = 0.020
retirement_investment_rate       = 0.100  # DOL/EBSA; Vanguard How America Saves

home_appreciation_rate_real     = home_appreciation_rate_nominal - inflation_rate   #  0.5%
personal_asset_growth_rate_real = personal_asset_growth_rate_nominal - inflation_rate  # -0.5%

mortgage_interest_rate       = 0.0646   # Freddie Mac PMMS April 2, 2026
mortgage_down_payment        = 0.10
mortgage_term_years          = 30
average_home_price_multiplier = 3.5     # Census Housing Affordability Data

salary_growth_factors = [1.0, 1.2, 1.5, 1.8]   # BLS age-earnings profile Table 7

# ── Color Scheme ──
plan_colors = {
    'IBR_2014':       '#FF6B6B',
    'IBR_pre2014':    '#C92A2A',
    'ICR':            '#4ECDC4',
    'PAYE':           '#1098AD',
    'SAVE_undergrad': '#9775FA',
    'SAVE_grad':      '#6741D9',
}

print("✓ All parameters loaded.")
print(f"  FPL single: ${fpl_single:,.0f} | FPL family of 4: ${fpl_family_of_4:,.0f}")
print(f"  Mortgage rate: {mortgage_interest_rate*100:.2f}% (Freddie Mac Apr 2, 2026)")
print(f"  Student loan debt: ${initial_student_loan_debt:,.0f} ± ${initial_student_loan_debt_se:,.0f}")


## Core Simulation Function

`simulate_wealth_with_idr()` runs a Monte Carlo simulation of 1,000,000 individuals over a 40-year career and returns an array of net worth values.

### MOE/SE Integration
Each uncertain parameter is drawn from a normal distribution at the individual level:
- **Income** → N(mean, SE) from BLS quarterly standard errors  
- **Homeownership rate** → N(rate, MOE/1.645), converting Census 90% CI to SE  
- **Student loan debt** → N($37,500, $2,000)  
- **Employment rate** → perturbed by N(0, ±0.015) per career stage  

This means the simulation captures *both* individual-level variation *and* measurement uncertainty from the underlying survey data.


In [ ]:
def simulate_wealth_with_idr(avg_income, factor, idr_settings,
                              home_rate, emp_rates, fpl_base,
                              income_se=0.0, home_rate_moe=0.0,
                              debt_mean=None, debt_se=0.0):
    """
    Simulate NET WORTH accumulation over 40-year career (age 22–62).

    Returns ARRAY of net worth values (length = num_individuals) in real 2025 $.
    Net Worth = (Savings + Home Equity + Retirement) − (Student Loans + Mortgage + Consumer Debt)
    """
    if debt_mean is None:
        debt_mean = initial_student_loan_debt

    adjusted_income = float(avg_income * factor)

    # Draw income with SE uncertainty
    if income_se > 0:
        individual_incomes = np.maximum(
            np.random.normal(adjusted_income, income_se, num_individuals), 0.0).astype(float)
    else:
        individual_incomes = np.full(num_individuals, adjusted_income, dtype=float)

    # Draw homeownership rate with MOE uncertainty (90% CI → SE = MOE / 1.645)
    if home_rate_moe > 0:
        sampled_home_rate = float(np.clip(
            np.random.normal(home_rate, home_rate_moe / 1.645), 0.0, 1.0))
    else:
        sampled_home_rate = home_rate

    # Draw student-loan debt with SE uncertainty
    if debt_se > 0:
        individual_debt = np.maximum(
            np.random.normal(debt_mean, debt_se, num_individuals), 0.0).astype(float)
    else:
        individual_debt = np.full(num_individuals, float(debt_mean), dtype=float)

    # Initialize assets & liabilities
    liquid_assets        = individual_incomes * np.random.uniform(0.1, 0.3, num_individuals)
    retirement_balance   = np.zeros(num_individuals, dtype=float)
    home_equity          = np.zeros(num_individuals, dtype=float)
    student_loan_balance = individual_debt.copy()
    mortgage_balance     = np.zeros(num_individuals, dtype=float)
    consumer_debt        = np.zeros(num_individuals, dtype=float)

    # Housing setup
    owns_home           = np.random.rand(num_individuals) < sampled_home_rate
    home_purchase_price = (individual_incomes * average_home_price_multiplier).astype(float)
    mortgage_balance[owns_home] = home_purchase_price[owns_home] * (1 - mortgage_down_payment)
    home_value          = home_purchase_price.copy()

    repayment_rate  = idr_settings['repayment_rate']
    repayment_years = idr_settings['years']
    fpl_threshold   = fpl_base * idr_settings['fpl_multiplier']
    salary_by_stage = [individual_incomes * x for x in salary_growth_factors]
    cumulative_years = 0

    for stage_idx, (emp_rate, salary, years_in_stage) in enumerate(
            zip(emp_rates, salary_by_stage, stage_durations)):

        # Employment with SE perturbation
        sampled_emp = float(np.clip(np.random.normal(emp_rate, employment_rates_se), 0.0, 1.0))
        employed     = np.random.rand(num_individuals) < sampled_emp
        annual_income = (employed.astype(float) * salary).astype(float)

        # IDR payment
        if cumulative_years < repayment_years:
            disc_income        = np.maximum(annual_income - fpl_threshold, 0.0)
            annual_idr_payment = (disc_income * repayment_rate).astype(float)
        else:
            annual_idr_payment = np.zeros(num_individuals, dtype=float)

        # Student loan balance
        sl_interest          = (student_loan_balance * student_loan_interest_rate).astype(float)
        student_loan_balance = np.maximum(
            student_loan_balance + sl_interest - annual_idr_payment, 0.0)

        # Mortgage payment
        if stage_idx == 0:
            monthly_rate  = mortgage_interest_rate / 12
            n_payments    = mortgage_term_years * 12
            monthly_pmt   = np.zeros(num_individuals, dtype=float)
            monthly_pmt[owns_home] = (
                mortgage_balance[owns_home] * monthly_rate *
                (1 + monthly_rate) ** n_payments /
                ((1 + monthly_rate) ** n_payments - 1))
            annual_mortgage_payment = (monthly_pmt * 12).astype(float)
        else:
            annual_mortgage_payment = np.where(
                owns_home & (mortgage_balance > 0),
                mortgage_balance * 0.08, 0.0).astype(float)

        # Mortgage balance
        mort_interest    = (mortgage_balance * mortgage_interest_rate).astype(float)
        mort_principal   = np.maximum(annual_mortgage_payment - mort_interest, 0.0)
        mortgage_balance = np.maximum(mortgage_balance - mort_principal, 0.0)

        # Home value & equity
        home_value[owns_home] = (
            home_value[owns_home] * (1 + home_appreciation_rate_real) ** years_in_stage)
        home_equity = np.maximum(home_value - mortgage_balance, 0.0)

        # Consumer debt (non-homeowners)
        consumer_debt[~owns_home] = (annual_income[~owns_home] * 0.05).astype(float)

        # Retirement (10% contrib → 7% real growth)
        retirement_balance = (
            (retirement_balance + annual_income * retirement_investment_rate) *
            (1 + 0.07) ** years_in_stage).astype(float)

        # Personal savings
        living_expenses = annual_income * 0.60
        available       = annual_income - living_expenses - annual_idr_payment - annual_mortgage_payment
        annual_savings  = np.maximum(available * 0.5, 0.0)
        liquid_assets   = (
            (liquid_assets + annual_savings * years_in_stage) *
            (1 + personal_asset_growth_rate_real) ** years_in_stage).astype(float)

        cumulative_years += years_in_stage

    # Loan forgiveness after repayment period
    if cumulative_years >= repayment_years:
        student_loan_balance = np.zeros(num_individuals, dtype=float)

    total_assets      = liquid_assets + retirement_balance + home_equity
    total_liabilities = student_loan_balance + mortgage_balance + consumer_debt
    return (total_assets - total_liabilities).astype(float)


def summarize(arr):
    """Return (mean, lower_95ci, upper_95ci) from simulation array."""
    m  = np.mean(arr)
    se = np.std(arr) / np.sqrt(len(arr))
    return m, m - 1.96 * se, m + 1.96 * se


def save_fig(fig, filename):
    """Save figure to output_dir if set, then display inline."""
    if output_dir:
        path = os.path.join(output_dir, filename)
        fig.savefig(path, dpi=150, bbox_inches='tight')
        print(f"Saved: {path}")
    plt.show()
    plt.close(fig)

print("✓ Simulation function defined.")


## Part 1 — Individual Scenarios (6 Race/Gender Groups)

**Timeline:** Age 22 (graduation) → Age 62 (retirement) = 40 years  
**Wealth:** Real 2025 dollars (inflation-adjusted)  
**Error bars:** 95% confidence interval from Monte Carlo distribution


In [ ]:
print("Running Part 1: Individual scenarios (1M individuals per cell)...")
results_by_plan = {}
for plan_name, settings in idr_plans.items():
    results_by_plan[plan_name] = {}
    for category, group_data in data.items():
        results_by_plan[plan_name][category] = {}
        for bracket, factor in zip(income_brackets, income_factors):
            results_by_plan[plan_name][category][bracket] = simulate_wealth_with_idr(
                group_data['avg_income'], factor, settings,
                home_purchase_rates[category],
                employment_rates[category],
                fpl_single,
                income_se=group_data['income_se'],
                home_rate_moe=home_purchase_rates_moe[category],
                debt_mean=initial_student_loan_debt,
                debt_se=initial_student_loan_debt_se,
            )
print("✓ Part 1 simulations complete.")


In [ ]:
# Figure 1: Net Worth at Retirement — by race/gender (with 95% CI error bars)
for category in data.keys():
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    x     = np.arange(len(income_brackets))
    width = 0.13

    for i, (plan_name, color) in enumerate(plan_colors.items()):
        means, lowers, uppers = zip(*[
            summarize(results_by_plan[plan_name][category][bracket])
            for bracket in income_brackets
        ])
        means  = list(means)
        errors = [m - l for m, l in zip(means, lowers)]

        bars = ax.bar(x + (i - 2.5) * width, means, width,
                      label=plan_name.replace('_', ' '), color=color,
                      edgecolor='white', linewidth=0.5)
        ax.errorbar(x + (i - 2.5) * width, means, yerr=errors,
                    fmt='none', color='#333333', capsize=3, linewidth=1, capthick=1)
        for j, bar in enumerate(bars):
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2,
                    h * (1.01 if i % 2 == 0 else 1.03),
                    f'${h/1000:.0f}K', ha='center', va='bottom', fontsize=7)

    avg_income = data[category]['avg_income']
    ax.set_title(
        f'{category}  |  Avg Income: ${avg_income:,} (BLS Q4 2024)\n'
        f'Net Worth at Retirement — Age 62, Real 2025 $  |  Error bars = 95% CI',
        fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(income_brackets, fontsize=10)
    ax.set_ylabel('Net Worth (Real 2025 $)', fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    save_fig(fig, f'fig1_individual_{category.lower().replace(" ","_")}.png')


## Part 2 — Family of 4 Scenarios (3 Racial Groups × 3 Income Tiers)

Family median incomes from Census CPS/ASEC 2023-2024, with MOE applied:
- **Black families:** $56,020 ± $323
- **White families:** $92,530 ± $224  
- **Hispanic families:** $70,950 ± $451

FPL threshold for family of 4: **$33,000** (HHS 2026)


In [ ]:
races      = list(family_income_by_race.keys())
tier_names = list(family_income_tiers.keys())

print("Running Part 2: Family of 4 scenarios...")
family_results = {}
for plan_name, settings in idr_plans.items():
    family_results[plan_name] = {}
    for race, base_income in family_income_by_race.items():
        family_results[plan_name][race] = {}
        for tier_name, factor in family_income_tiers.items():
            family_results[plan_name][race][tier_name] = simulate_wealth_with_idr(
                base_income, factor, settings,
                home_purchase_rates_by_race[race],
                employment_rates_by_race[race],
                fpl_family_of_4,
                income_se=family_income_moe[race],
                home_rate_moe=home_purchase_rates_moe_by_race[race],
                debt_mean=initial_student_loan_debt,
                debt_se=initial_student_loan_debt_se,
            )
print("✓ Part 2 simulations complete.")


In [ ]:
# Figure 2: Family net worth by race (with 95% CI error bars)
for race in races:
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    x     = np.arange(len(tier_names))
    width = 0.13

    for i, (plan_name, color) in enumerate(plan_colors.items()):
        means, lowers, uppers = zip(*[
            summarize(family_results[plan_name][race][tier])
            for tier in tier_names
        ])
        means  = list(means)
        errors = [m - l for m, l in zip(means, lowers)]

        bars = ax.bar(x + (i - 2.5) * width, means, width,
                      label=plan_name.replace('_', ' '), color=color,
                      edgecolor='white', linewidth=0.5)
        ax.errorbar(x + (i - 2.5) * width, means, yerr=errors,
                    fmt='none', color='#333333', capsize=3, linewidth=1, capthick=1)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2,
                    h * (1.01 if i % 2 == 0 else 1.03),
                    f'${h/1000:.0f}K', ha='center', va='bottom', fontsize=7)

    base_inc = family_income_by_race[race]
    ax.set_title(
        f'{race} Family of 4  |  Median HH Income: ${base_inc:,} (Census CPS/ASEC 2024)\n'
        f'Net Worth at Retirement — Age 62, Real 2025 $  |  Error bars = 95% CI',
        fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(tier_names, fontsize=10)
    ax.set_ylabel('Net Worth (Real 2025 $)', fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    save_fig(fig, f'fig2_family_{race.lower()}.png')


## Part 3 — Cross-Racial Comparison Charts

In [ ]:
plan_names_short = [p.replace('_', ' ') for p in idr_plans.keys()]
print("Generating Part 3: Cross-racial comparison charts...")

# Figure 3: Annual IDR Payment Burden
for race in races:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    for ax, (tier_name, tier_factor) in zip(axes, family_income_tiers.items()):
        annual_income = family_income_by_race[race] * tier_factor
        x = np.arange(len(idr_plans))
        payments = [max(annual_income - fpl_family_of_4 * s['fpl_multiplier'], 0) * s['repayment_rate']
                    for s in idr_plans.values()]
        bars = ax.bar(x, payments, color=list(plan_colors.values()), edgecolor='white', linewidth=0.8)
        for i, bar in enumerate(bars):
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h*(1.02 if i%2==0 else 1.05),
                        f'${h/1000:.1f}K', ha='center', va='bottom', fontsize=7)
            else:
                ax.text(bar.get_x() + bar.get_width()/2, 200, '$0',
                        ha='center', va='bottom', fontsize=7, color='gray')
        ax.set_title(f'{tier_name}\n(${annual_income:,.0f})', fontsize=10, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(plan_names_short, fontsize=8, rotation=20, ha='right')
        ax.set_ylabel('Annual IDR Payment ($)' if ax == axes[0] else '', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
    fig.suptitle(f'{race} Family of 4 — Annual IDR Payment Burden',
                 fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    save_fig(fig, f'fig3_idr_payment_{race.lower()}.png')

print("✓ Figure 3 saved.")


In [ ]:
# Figure 4: Post-IDR Disposable Income
for race in races:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    for ax, (tier_name, tier_factor) in zip(axes, family_income_tiers.items()):
        annual_income = family_income_by_race[race] * tier_factor
        x = np.arange(len(idr_plans))
        disposable = [annual_income - max(annual_income - fpl_family_of_4*s['fpl_multiplier'],0)*s['repayment_rate']
                      for s in idr_plans.values()]
        bars = ax.bar(x, disposable, color=list(plan_colors.values()), edgecolor='white', linewidth=0.8)
        for i, bar in enumerate(bars):
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h*(1.01 if i%2==0 else 1.02),
                    f'${h/1000:.0f}K', ha='center', va='bottom', fontsize=7)
        ax.set_title(f'{tier_name}\n(${annual_income:,.0f})', fontsize=10, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(plan_names_short, fontsize=8, rotation=20, ha='right')
        ax.set_ylabel('Post-IDR Disposable Income ($)' if ax == axes[0] else '', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
    fig.suptitle(f'{race} Family of 4 — Post-IDR Disposable Income',
                 fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    save_fig(fig, f'fig4_disposable_income_{race.lower()}.png')

print("✓ Figure 4 saved.")


In [ ]:
# Figure 5: Monthly Payment & % of Income
for race in races:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharey='row')
    colors_list = list(plan_colors.values())
    for ax, (tier_name, tier_factor) in zip(axes[0], family_income_tiers.items()):
        annual_income = family_income_by_race[race] * tier_factor
        x = np.arange(len(idr_plans))
        monthly = [max(annual_income - fpl_family_of_4*s['fpl_multiplier'],0)*s['repayment_rate']/12
                   for s in idr_plans.values()]
        bars = ax.bar(x, monthly, color=colors_list, edgecolor='white', linewidth=0.8)
        for i, bar in enumerate(bars):
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x()+bar.get_width()/2, h*(1.03 if i%2==0 else 1.07),
                        f'${h:.0f}', ha='center', va='bottom', fontsize=7)
            else:
                ax.text(bar.get_x()+bar.get_width()/2, 10, '$0',
                        ha='center', va='bottom', fontsize=7, color='gray')
        ax.set_title(f'{tier_name}\n(${annual_income:,.0f})', fontsize=9, fontweight='bold')
        ax.set_xticks(x); ax.set_xticklabels(plan_names_short, fontsize=7, rotation=20, ha='right')
        ax.set_ylabel('Monthly Payment ($)' if ax==axes[0][0] else '', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
    for ax, (tier_name, tier_factor) in zip(axes[1], family_income_tiers.items()):
        annual_income = family_income_by_race[race] * tier_factor
        x = np.arange(len(idr_plans))
        pct = [max(annual_income-fpl_family_of_4*s['fpl_multiplier'],0)*s['repayment_rate']/annual_income*100
               for s in idr_plans.values()]
        bars = ax.bar(x, pct, color=colors_list, edgecolor='white', linewidth=0.8)
        for i, bar in enumerate(bars):
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x()+bar.get_width()/2, h*(1.05 if i%2==0 else 1.12),
                        f'{h:.1f}%', ha='center', va='bottom', fontsize=7)
            else:
                ax.text(bar.get_x()+bar.get_width()/2, 0.3, '0%',
                        ha='center', va='bottom', fontsize=7, color='gray')
        ax.set_title(f'{tier_name}', fontsize=9, fontweight='bold')
        ax.set_xticks(x); ax.set_xticklabels(plan_names_short, fontsize=7, rotation=20, ha='right')
        ax.set_ylabel('% of Gross Income' if ax==axes[1][0] else '', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
    fig.suptitle(f'{race} Family of 4 — Monthly Payment (Top) & % of Income (Bottom)',
                 fontsize=13, fontweight='bold', y=0.995)
    plt.tight_layout()
    save_fig(fig, f'fig5_monthly_payment_{race.lower()}.png')

print("✓ Figure 5 saved.")


In [ ]:
# Figure 6: Wealth Generation with CI
for race in races:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    colors_list = list(plan_colors.values())
    for ax, tier_name in zip(axes, tier_names):
        x = np.arange(len(idr_plans))
        means, lowers, uppers = zip(*[
            summarize(family_results[plan_name][race][tier_name])
            for plan_name in idr_plans])
        means  = list(means)
        errors = [m-l for m,l in zip(means, lowers)]
        bars = ax.bar(x, means, color=colors_list, edgecolor='white', linewidth=0.8)
        ax.errorbar(x, means, yerr=errors, fmt='none', color='#333333',
                    capsize=3, linewidth=1, capthick=1)
        for i, bar in enumerate(bars):
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h*(1.01 if i%2==0 else 1.03),
                    f'${h/1000:.0f}K', ha='center', va='bottom', fontsize=7)
        ax.set_title(f'{tier_name}', fontsize=10, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(plan_names_short, fontsize=8, rotation=20, ha='right')
        ax.set_ylabel('Net Worth (Real 2025 $)' if ax==axes[0] else '', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
    fig.suptitle(f'{race} Family of 4 — Net Worth at Retirement by IDR Plan\n(Error bars = 95% CI)',
                 fontsize=14, fontweight='bold', y=1.03)
    plt.tight_layout()
    save_fig(fig, f'fig6_wealth_generation_{race.lower()}.png')

print("✓ Figure 6 saved.")


In [ ]:
# Figure 7: Racial Wealth Gap
for minority_race in ['Black', 'Hispanic']:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    colors_list = list(plan_colors.values())
    for ax, tier_name in zip(axes, tier_names):
        x = np.arange(len(idr_plans))
        white_means = [np.mean(family_results[p]['White'][tier_name]) for p in idr_plans]
        race_means  = [np.mean(family_results[p][minority_race][tier_name]) for p in idr_plans]
        gap_pct     = [(r/w - 1)*100 for r,w in zip(race_means, white_means)]
        bars = ax.bar(x, gap_pct, color=colors_list, edgecolor='white', linewidth=0.8)
        for i, bar in enumerate(bars):
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h*1.08,
                    f'{h:.1f}%', ha='center',
                    va='top' if h < 0 else 'bottom', fontsize=8, fontweight='bold')
        ax.axhline(0, color='#333333', linewidth=1.5, linestyle='--')
        ax.set_title(f'{tier_name}', fontsize=10, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(plan_names_short, fontsize=8, rotation=20, ha='right')
        ax.set_ylabel('Wealth Gap vs White (%)' if ax==axes[0] else '', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
    fig.suptitle(
        f'{minority_race} vs White — Wealth Gap by IDR Plan\n'
        f'(Negative % = {minority_race} family accumulates less wealth than White family)',
        fontsize=13, fontweight='bold', y=1.03)
    plt.tight_layout()
    save_fig(fig, f'fig7_wealth_gap_{minority_race.lower()}.png')

print("✓ Figure 7 saved.")


## Part 4 — Sensitivity Analysis

Three components:
1. **Tornado Chart** (Fig 8) — One-at-a-time ±1 SE/MOE perturbation showing which parameters drive the most outcome uncertainty
2. **Economic Scenario Matrix** (Fig 9) — Pessimistic / Baseline / Optimistic crossed with all 6 IDR plans
3. **Racial Wealth Gap Sensitivity** (Fig 10) — Does the gap widen or narrow by scenario?

**Reference case for tornado:** White Men, Median income, IBR_2014


In [ ]:
print("Running Part 4: Sensitivity Analysis...")

ref_category  = 'White Men'
ref_bracket   = 'Median 50%'
ref_plan      = 'IBR_2014'
ref_plan_set  = idr_plans[ref_plan]
ref_income    = data[ref_category]['avg_income']
ref_income_se = data[ref_category]['income_se']
ref_home_rate = home_purchase_rates[ref_category]
ref_home_moe  = home_purchase_rates_moe[ref_category]
ref_factor    = income_factors[income_brackets.index(ref_bracket)]

def run_ref(income_override=None, home_override=None,
            debt_override=None, mort_rate_override=None,
            ret_rate_override=None, emp_se_override=None):
    global mortgage_interest_rate, retirement_investment_rate, employment_rates_se
    orig_mort = mortgage_interest_rate
    orig_ret  = retirement_investment_rate
    orig_emp  = employment_rates_se
    if mort_rate_override is not None: mortgage_interest_rate = mort_rate_override
    if ret_rate_override  is not None: retirement_investment_rate = ret_rate_override
    if emp_se_override    is not None: employment_rates_se = emp_se_override
    result = simulate_wealth_with_idr(
        income_override if income_override is not None else ref_income,
        ref_factor, ref_plan_set,
        home_override if home_override is not None else ref_home_rate,
        employment_rates[ref_category], fpl_single,
        income_se=ref_income_se, home_rate_moe=ref_home_moe,
        debt_mean=debt_override if debt_override is not None else initial_student_loan_debt,
        debt_se=initial_student_loan_debt_se,
    )
    mortgage_interest_rate = orig_mort
    retirement_investment_rate = orig_ret
    employment_rates_se = orig_emp
    return np.mean(result)

baseline_nw = run_ref()
print(f"  Baseline net worth ({ref_category}, {ref_bracket}, {ref_plan}): ${baseline_nw/1000:.0f}K")

sensitivity_params = [
    ('Annual Income',          ref_income - ref_income_se,        ref_income + ref_income_se,        'income'),
    ('Homeownership Rate',     ref_home_rate - ref_home_moe,      ref_home_rate + ref_home_moe,      'home'),
    ('Initial Loan Debt',      initial_student_loan_debt - 2000,  initial_student_loan_debt + 2000,  'debt'),
    ('Mortgage Rate',          mortgage_interest_rate - 0.005,    mortgage_interest_rate + 0.005,    'mort'),
    ('Retirement Contrib.',    retirement_investment_rate - 0.01, retirement_investment_rate + 0.01, 'ret'),
    ('Employment Rate Uncert.',employment_rates_se - 0.005,       employment_rates_se + 0.005,       'emp'),
]

tornado_results = []
for label, low_val, high_val, param in sensitivity_params:
    dispatch = {
        'income': lambda l, h: (run_ref(income_override=l), run_ref(income_override=h)),
        'home':   lambda l, h: (run_ref(home_override=max(l,0)), run_ref(home_override=min(h,1))),
        'debt':   lambda l, h: (run_ref(debt_override=max(l,0)), run_ref(debt_override=h)),
        'mort':   lambda l, h: (run_ref(mort_rate_override=max(l,0.01)), run_ref(mort_rate_override=h)),
        'ret':    lambda l, h: (run_ref(ret_rate_override=max(l,0)), run_ref(ret_rate_override=h)),
        'emp':    lambda l, h: (run_ref(emp_se_override=max(l,0)), run_ref(emp_se_override=h)),
    }
    nw_low, nw_high = dispatch[param](low_val, high_val)
    tornado_results.append((label, nw_low - baseline_nw, nw_high - baseline_nw))

tornado_results.sort(key=lambda r: abs(r[2] - r[1]))
print("✓ Tornado parameters computed.")


In [ ]:
# Figure 8: Tornado Chart
labels = [r[0] for r in tornado_results]
lows   = [r[1] for r in tornado_results]
highs  = [r[2] for r in tornado_results]

fig_t, ax_t = plt.subplots(figsize=(11, 6))
y_pos = np.arange(len(labels))

for i, (label, low, high) in enumerate(zip(labels, lows, highs)):
    left      = min(low, high)
    right     = max(low, high)
    width_bar = right - left
    ax_t.barh(i, width_bar, left=left,
              color='#4ECDC4' if high > 0 else '#FF6B6B',
              edgecolor='white', linewidth=0.5)
    ax_t.text(left - 2000,  i, f'${left/1000:+.0f}K',  va='center', ha='right', fontsize=8, color='#333')
    ax_t.text(right + 2000, i, f'${right/1000:+.0f}K', va='center', ha='left',  fontsize=8, color='#333')

ax_t.axvline(0, color='#222', linewidth=1.5, linestyle='--')
ax_t.set_yticks(y_pos)
ax_t.set_yticklabels(labels, fontsize=10)
ax_t.set_xlabel('Change in Mean Net Worth vs. Baseline (Real 2025 $)', fontsize=10)
ax_t.set_title(
    f'Sensitivity (Tornado) Chart — {ref_category}, {ref_bracket} Income, {ref_plan.replace("_"," ")}\n'
    f'Baseline: ${baseline_nw/1000:.0f}K  |  Each bar = ±1 SE/MOE perturbation',
    fontsize=11, fontweight='bold')
ax_t.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_fig(fig_t, 'fig8_sensitivity_tornado.png')
print("✓ Figure 8 (Tornado) saved.")


In [ ]:
# Figure 9: Economic Scenario Matrix
scenarios = {
    'Pessimistic': {'income_mult': 0.95, 'mortgage_rate': 0.075,
                    'home_appreciation': 0.01, 'retirement_return': 0.05, 'loan_debt': 42000},
    'Baseline':    {'income_mult': 1.00, 'mortgage_rate': 0.0646,
                    'home_appreciation': 0.03, 'retirement_return': 0.07, 'loan_debt': 37500},
    'Optimistic':  {'income_mult': 1.05, 'mortgage_rate': 0.055,
                    'home_appreciation': 0.04, 'retirement_return': 0.09, 'loan_debt': 30000},
}

def run_scenario(plan_name, sp):
    global mortgage_interest_rate, home_appreciation_rate_real, home_appreciation_rate_nominal
    orig_m, orig_r, orig_n = mortgage_interest_rate, home_appreciation_rate_real, home_appreciation_rate_nominal
    mortgage_interest_rate         = sp['mortgage_rate']
    home_appreciation_rate_nominal = sp['home_appreciation']
    home_appreciation_rate_real    = sp['home_appreciation'] - inflation_rate
    result = simulate_wealth_with_idr(
        data['White Men']['avg_income'] * sp['income_mult'], 1.0, idr_plans[plan_name],
        home_purchase_rates['White Men'], employment_rates['White Men'], fpl_single,
        income_se=data['White Men']['income_se'],
        home_rate_moe=home_purchase_rates_moe['White Men'],
        debt_mean=sp['loan_debt'], debt_se=initial_student_loan_debt_se,
    )
    mortgage_interest_rate         = orig_m
    home_appreciation_rate_real    = orig_r
    home_appreciation_rate_nominal = orig_n
    return np.mean(result)

scenario_grid = {s: {p: run_scenario(p, sp) for p in idr_plans} for s, sp in scenarios.items()}

fig_s, ax_s = plt.subplots(figsize=(13, 6))
x_pos = np.arange(len(idr_plans))
plan_list    = list(idr_plans.keys())
scen_colors  = {'Pessimistic': '#E74C3C', 'Baseline': '#3498DB', 'Optimistic': '#2ECC71'}

for i, (scen_name, scen_vals) in enumerate(scenario_grid.items()):
    means = [scen_vals[p] for p in plan_list]
    bars  = ax_s.bar(x_pos + (i - 1) * 0.25, means, 0.25,
                     label=scen_name, color=scen_colors[scen_name], edgecolor='white', linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        ax_s.text(bar.get_x()+bar.get_width()/2, h*1.01, f'${h/1000:.0f}K',
                  ha='center', va='bottom', fontsize=7)

ax_s.set_xticks(x_pos)
ax_s.set_xticklabels([p.replace('_',' ') for p in plan_list], fontsize=9, rotation=15, ha='right')
ax_s.set_ylabel('Mean Net Worth at Retirement (Real 2025 $)', fontsize=10)
ax_s.set_title(
    'Sensitivity Analysis — Economic Scenarios × IDR Plan\n'
    'Reference: White Men, Median Income  |  Pessimistic / Baseline / Optimistic',
    fontsize=11, fontweight='bold')
ax_s.legend(fontsize=10)
ax_s.grid(axis='y', alpha=0.3)
ax_s.text(0.01, 0.01,
    "Pessimistic: income −5%, mortgage 7.5%, home appr. 1%, ret. return 5%, debt $42K\n"
    "Baseline:    income base, mortgage 6.46%, home appr. 3%, ret. return 7%, debt $37.5K\n"
    "Optimistic:  income +5%, mortgage 5.5%, home appr. 4%, ret. return 9%, debt $30K",
    transform=ax_s.transAxes, fontsize=7.5, verticalalignment='bottom',
    bbox=dict(boxstyle='round', facecolor='#F8F9FA', alpha=0.8))
plt.tight_layout()
save_fig(fig_s, 'fig9_sensitivity_scenarios.png')
print("✓ Figure 9 (Scenario Matrix) saved.")


In [ ]:
# Figure 10: Racial Wealth Gap Sensitivity
def run_race_scenario(race, plan_name, sp):
    global mortgage_interest_rate, home_appreciation_rate_real, home_appreciation_rate_nominal
    orig_m, orig_r, orig_n = mortgage_interest_rate, home_appreciation_rate_real, home_appreciation_rate_nominal
    mortgage_interest_rate         = sp['mortgage_rate']
    home_appreciation_rate_nominal = sp['home_appreciation']
    home_appreciation_rate_real    = sp['home_appreciation'] - inflation_rate
    result = simulate_wealth_with_idr(
        family_income_by_race[race] * sp['income_mult'], 1.0, idr_plans[plan_name],
        home_purchase_rates_by_race[race], employment_rates_by_race[race], fpl_family_of_4,
        income_se=family_income_moe[race],
        home_rate_moe=home_purchase_rates_moe_by_race[race],
        debt_mean=sp['loan_debt'], debt_se=initial_student_loan_debt_se,
    )
    mortgage_interest_rate         = orig_m
    home_appreciation_rate_real    = orig_r
    home_appreciation_rate_nominal = orig_n
    return np.mean(result)

rep_plan = 'IBR_2014'
gap_by_scenario = {}
for scen_name, scen_params in scenarios.items():
    white_nw = run_race_scenario('White', rep_plan, scen_params)
    gap_by_scenario[scen_name] = {
        race: (run_race_scenario(race, rep_plan, scen_params) / white_nw - 1) * 100
        for race in ['Black', 'Hispanic']
    }

fig_g, ax_g = plt.subplots(figsize=(9, 5))
race_min_colors = {'Black': '#E74C3C', 'Hispanic': '#2ECC71'}
scen_labels     = list(scenarios.keys())
x_g = np.arange(len(scen_labels))

for i, (race, color) in enumerate(race_min_colors.items()):
    gaps = [gap_by_scenario[s][race] for s in scen_labels]
    bars = ax_g.bar(x_g + (i - 0.5) * 0.30, gaps, 0.30,
                    label=race, color=color, edgecolor='white', linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        ax_g.text(bar.get_x()+bar.get_width()/2,
                  h*1.05 if h < 0 else h*1.02, f'{h:.1f}%',
                  ha='center', va='top' if h < 0 else 'bottom', fontsize=9, fontweight='bold')

ax_g.axhline(0, color='#333', linewidth=1.5, linestyle='--')
ax_g.set_xticks(x_g)
ax_g.set_xticklabels(scen_labels, fontsize=11)
ax_g.set_ylabel('Wealth Gap vs White Family (%)', fontsize=10)
ax_g.set_title(
    f'Racial Wealth Gap Sensitivity by Economic Scenario\n'
    f'Family of 4, {rep_plan.replace("_"," ")} Plan — Median Income Tier',
    fontsize=11, fontweight='bold')
ax_g.legend(fontsize=10)
ax_g.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_fig(fig_g, 'fig10_wealth_gap_sensitivity.png')
print("✓ Figure 10 (Racial Wealth Gap Sensitivity) saved.")


## Summary

All 26 figures generated. Files saved to `output_dir` (if set) and displayed inline above.

| Part | Figures | Description |
|---|---|---|
| 1 | Fig 1 (×6) | Individual net worth at retirement by race/gender |
| 2 | Fig 2 (×3) | Family of 4 net worth by race |
| 3 | Fig 3–7 (×14) | Cross-racial comparisons: payment burden, disposable income, monthly payment, wealth generation, wealth gap |
| 4 | Fig 8–10 (×3) | Sensitivity: tornado chart, scenario matrix, racial gap sensitivity |

### Key Updates vs. Previous Version
- **FPL:** $15,960 / $33,000 (2026 HHS)
- **Income data:** BLS Q4 2024 (most recent complete quarter)
- **Student loan debt:** $37,500 ± $2,000 (Education Data Initiative 2025)
- **Mortgage rate:** 6.46% (Freddie Mac PMMS April 2, 2026)
- **MOE/SE:** All stochastic parameters now draw from N(mean, SE) at the individual level
- **Sensitivity analysis:** 3 new figures (Part 4)
